In [ ]:
import os
import time

from bng_simulator.utils.services_utils import send_request
from bng_simulator.utils.io_dict_utils import(
    save_yaml,
    round_dict_values,
    build_tree_from_dict,
    convert_tree_into_proper_dict
)

In [ ]:
# Get config, scenario , infos
scenario_infos = send_request("get_sim_config")

In [ ]:
# Print the vehicles in the environment
for v, v_info in scenario_infos["vehicles"].items():
    _infos = {_k : _v for _k, _v in v_info.items() if _k not in ["scenario_args", "sensors"]}
    print(f"Vehicle {v} :")
    for _k, _v in _infos.items():
        print(f"\t{_k} : {_v}")

In [ ]:
# Print the part numbers, used as unique identifiers for the vehicle components
for v_name, v_part in scenario_infos["vehicles_part"].items():
    print(f"Vehicle {v_name} :  {v_part}")

In [ ]:
# Let's enumerate all gear options and their characteristics
def extract_gear_infos(vehicle_name = None):
    """
    Go through all gears and extract their characteristics
    """
    in_args = {}
    in_args_set = {}
    if vehicle_name is not None:
        in_args["vehicle_name"] = vehicle_name
        in_args_set["vehicle_name"] = vehicle_name
    # Let's first get current infos
    gear_infos = send_request("get_gearbox_info", in_args)
    min_gear = int(gear_infos["minGearIndex"])
    max_gear = int(gear_infos["maxGearIndex"])
    # TODO: The min is actually negative of the actual min cause of abs.
    print(f"Min/max gear: {min_gear}/{max_gear}")
    
    # A function to parse the gear infos
    def parse_gear_infos(gear_infos):
        curr_gear = int(gear_infos["gearIndex"])
        gear_mode_index = gear_infos["gearModeIndex"]
        gear_ratio = gear_infos["gearRatio"]
        gear_name = gear_infos["gearName"]
        gear_mode = gear_infos["mode"]
        return gear_name, \
            {"index": curr_gear, "mode_index": gear_mode_index, 
             "gear_ratio": gear_ratio, "mode": gear_mode}
    
    # Store the different gear ratio
    mapping_info = {}
    gear_name, gear_info = parse_gear_infos(gear_infos)
    print(f"Gear {gear_name} - {gear_info}")
    initial_gear_name = gear_name
    for i in range(-1, max_gear + 3): # Just extra for the sake of it
        in_args_set["gear_index"] = i
        send_request("set_gearbox_index", in_args_set)
        # Wait for a bit
        time.sleep(0.5)
        # Get the gear infos
        gear_infos = send_request("get_gearbox_info", in_args)
        gear_name, gear_info = parse_gear_infos(gear_infos)
        gear_info["indexCmd"] = i
        print(f"Gear {gear_name} - {gear_info}")
        if gear_name not in mapping_info:
            mapping_info[gear_name] = gear_info
    # Set the initial gear index
    initial_gear_index = mapping_info[initial_gear_name]["indexCmd"]
    in_args_set["gear_index"] = initial_gear_index
    send_request("set_gearbox_index", in_args_set)
    return mapping_info

In [ ]:
# Extract the gear infos
gear_infos = extract_gear_infos()

In [ ]:
gear_infos

In [ ]:
# Useful functions to format the powertrain structure
def format_powertrain_structure(data, relevant_keys):
    """
    Format the powertrain structure.
    
    Args:
        data (Dict[str, Any]): The data.
        relevant_keys (Tuple[str]): The relevant keys.
        
    Returns:
        Dict[str, Any]: The tree-like structure.
    """
    roots, tree = build_tree_from_dict(data)
    res = {}
    for root in roots:
        res[root] = convert_tree_into_proper_dict(root, tree, data, relevant_keys)
    # # Let;s sort it
    # res = dict
    return round_dict_values(res)

powertrain_prop = send_request("get_powertrain_properties")
powertrain_infos = format_powertrain_structure(
    powertrain_prop,
    ['type', 'gearRatio', 'gearRatios', 'availableModes', 'diffTorqueSplit']
)

In [ ]:
powertrain_infos

In [ ]:
# Get controllers infos
controllers_infos = send_request("get_controller_infos")
controllers_infos = dict(sorted(controllers_infos.items()))

In [ ]:
controllers_infos

In [ ]:
# Kinematics infos
kin_args = {
    "world_space": False
}
kin_props = send_request("get_vehicle_properties", kin_args)

In [ ]:
# Handle engine infos
engine_infos = send_request("get_engine_infos")

In [ ]:
engine_infos

In [ ]:
kin_props

In [ ]:
# Let's try to output all these infos in a yaml file
vehicle_config_folder = "../config/vehicles/"
veh_name = "ego"
vehicle_path = vehicle_config_folder + scenario_infos["vehicles_part"][veh_name] + ".yaml"

final_dict_params = {
    **round_dict_values(kin_props),
    "engine" : round_dict_values(engine_infos),
    "gearbox": gear_infos,
    "powertrain": round_dict_values(powertrain_infos),
    "controllers": controllers_infos,
}
save_yaml(
    final_dict_params,
    vehicle_path,
    sort_keys=False
)